# FreetownRoad-6: training on Colab

1. **Runtime > Change runtime type > T4 GPU**
2. Put the project folder in Google Drive (e.g. `MyDrive/freetown-traffic`) containing `scripts/` and your split dataset in `data/freetownroad6/` (made by `scripts/split_by_location.py`).
3. Run the cells top to bottom. Training results are written to Drive, so a disconnect does not lose them.

In [ ]:
!nvidia-smi | head -n 15

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/freetown-traffic'   # change if you put it elsewhere
%cd $PROJECT

In [ ]:
!pip install -q ultralytics pyyaml pandas matplotlib

## Check the dataset
You should see six classes, and train/val/test image counts that match your split.

In [ ]:
!cat data/freetownroad6/data.yaml
!for s in train val test; do echo $s: $(ls data/freetownroad6/images/$s | wc -l) images; done

## Train
`--model yolov8n.pt` fine-tunes from COCO weights. To start from a model pre-trained on the Akujobi keke subset, pass that run's `best.pt` instead.

If Colab disconnects, re-run the setup cells and use the resume cell below.

In [ ]:
!python scripts/train.py --data data/freetownroad6/data.yaml --model yolov8n.pt \
    --imgsz 416 --batch 8 --epochs 100 --patience 20 \
    --project outputs/runs --name ft6_coco --export-to models

In [ ]:
# Resume an interrupted run (uncomment)
# !python scripts/train.py --resume outputs/runs/ft6_coco/weights/last.pt

## Evaluate on the held-out test locations
This runs on Colab's CPU, exactly as it will on your laptop.

In [ ]:
!python scripts/evaluate.py --model models/ft6_coco.pt --data data/freetownroad6/data.yaml
!python scripts/evaluate_baseline.py --data data/freetownroad6/data.yaml

## Download
`models/ft6_coco.pt` is the file to copy into the `models/` folder on your laptop. Results are in `outputs/eval/` and `outputs/metrics/` (all already in Drive).

In [ ]:
from google.colab import files
files.download('models/ft6_coco.pt')